In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
from flax.nnx import bridge


from positional_encoding import TimeSeriesPositionalEncoding
from graph_modules import Grand, GrandDiffuser
from jraphx.nn.conv import TransformerConv

from attention import MultiHeadAttention
from utils import create_v_model


class MLP(nnx.Module):
    def __init__(self, din: int, dmid: int, dout: int, *, rngs: nnx.Rngs):
        self.linear1 = nnx.Linear(din, dmid, rngs=rngs)
        self.linear2 = nnx.Linear(dmid, dout, rngs=rngs)

    def __call__(self, x: jax.Array):
        x = nnx.gelu(self.linear1(x))
        return self.linear2(x)


class Transformer(nnx.Module):

    def __init__(
        self,
        in_dim,
        out_dim,
        model_dim,
        num_heads,
        head_dim,
        *,
        rngs: nnx.Rngs,
        need_pos_enc=True
    ):
        self.need_pos_enc = need_pos_enc

        self.pos_enc = TimeSeriesPositionalEncoding()

        self.mha = MultiHeadAttention(
            in_dim=model_dim,
            out_dim=model_dim,
            num_heads=num_heads,
            head_dim=head_dim,
            rngs=rngs,
        )

        self.init_lin = nnx.Linear(in_dim, model_dim, rngs=rngs)

        self.ln1 = nnx.LayerNorm(model_dim, rngs=rngs)
        self.ln2 = nnx.LayerNorm(model_dim, rngs=rngs)

        self.mlp = MLP(model_dim, model_dim, out_dim, rngs=rngs)
        self.rngs = nnx.Rngs(rngs.params())

    def __call__(self, x):

        x = self.init_lin(x)

        if self.need_pos_enc:
            x = self.pos_enc(x)

        h = self.ln1(x)
        h = self.mha(h, h, h)
        x = x + h

        h = self.ln2(x)
        h = self.mlp(h)
        x = x + h

        return x


class STFormerBlock(nnx.Module):

    def __init__(
        self,
        in_dim,
        out_dim,
        model_dim,
        num_heads,
        head_dim,
        num_chanels,
        *,
        rngs: nnx.Rngs
    ):

        self.spatial_transformer = Transformer(
            in_dim=model_dim,
            out_dim=model_dim,
            num_heads=num_heads,
            model_dim=model_dim,
            head_dim=head_dim,
            need_pos_enc=False,
            rngs=rngs,
        )

        self.ln = nnx.LayerNorm(model_dim, rngs=rngs)

        self.mlp = MLP(model_dim, model_dim, out_dim, rngs=rngs)

        backup = nnx.split_rngs(rngs, splits=num_chanels, only="params")

        self.temporal_transformer = create_v_model(
            rngs,
            Transformer,
            model_args={
                "in_dim": in_dim,
                "out_dim": model_dim,
                "num_heads": num_heads,
                "model_dim": model_dim,
                "num_heads": num_heads,
                "head_dim": head_dim,
                "need_pos_enc": True,
            },
        )

        nnx.restore_rngs(backup)

        self.rngs = nnx.Rngs(rngs.params())

    def __call__(self, x):

        B, S, C, D = x.shape

        h = nnx.vmap(lambda model, x: model(x), in_axes=(0, 2), out_axes=2)(
            self.temporal_transformer, x
        )

        h = h.reshape(B * S, C, D)

        h = self.spatial_transformer(h).reshape(B, S, C, D)

        res = h
        h = self.ln(h)
        h = self.mlp(h)

        return h + res


class DiffGraphSTFormerBlock(nnx.Module):

    def __init__(
        self,
        in_dim,
        out_dim,
        model_dim,
        num_heads,
        head_dim,
        num_chanels,
        *,
        rngs: nnx.Rngs
    ):

        self.spatial_model = Grand(
            in_dim=model_dim,
            model_dim=model_dim,
            out_dim=model_dim,
            num_heads=num_heads,
            head_dim=head_dim,
            num_chanels=num_chanels,
            rngs=rngs,
        )

        self.ln = nnx.LayerNorm(model_dim, rngs=rngs)

        self.mlp = MLP(model_dim, model_dim, out_dim, rngs=rngs)

        backup = nnx.split_rngs(rngs, splits=num_chanels, only="params")

        self.temporal_transformer = create_v_model(
            rngs,
            Transformer,
            model_args={
                "in_dim": in_dim,
                "out_dim": model_dim,
                "model_dim": model_dim,
                "num_heads": num_heads,
                "head_dim": head_dim,
                "need_pos_enc": True,
            },
        )

        nnx.restore_rngs(backup)

        self.rngs = nnx.Rngs(rngs.params())

    def __call__(self, x: jax.Array) -> jax.Array:
        _, S, _, _ = x.shape

        h = nnx.vmap(lambda model, x: model(x), in_axes=(0, 2), out_axes=2)(
            self.temporal_transformer, x
        )

        t_grid = jnp.linspace(0, 0.5, 4)

        h = self.spatial_model(h, t_grid)

        res = h
        h = self.ln(h)
        out = self.mlp(h)

        return out + res


class TFormerBlock(nnx.Module):

    def __init__(
        self,
        in_dim,
        out_dim,
        model_dim,
        num_heads,
        head_dim,
        num_chanels,
        *,
        rngs: nnx.Rngs
    ):

        self.spatial_transformer = Transformer(
            in_dim=model_dim,
            out_dim=model_dim,
            num_heads=num_heads,
            model_dim=model_dim,
            head_dim=head_dim,
            need_pos_enc=False,
            rngs=rngs,
        )

        self.ln = nnx.LayerNorm(model_dim, rngs=rngs)

        self.mlp = MLP(model_dim, model_dim, out_dim, rngs=rngs)

        backup = nnx.split_rngs(rngs, splits=num_chanels, only="params")

        self.temporal_transformer = create_v_model(
            rngs,
            Transformer,
            model_args={
                "in_dim": in_dim,
                "out_dim": model_dim,
                "num_heads": num_heads,
                "model_dim": model_dim,
                "num_heads": num_heads,
                "head_dim": head_dim,
                "need_pos_enc": True,
            },
        )

        nnx.restore_rngs(backup)

        self.rngs = nnx.Rngs(rngs.params())

    def __call__(self, x):

        h = nnx.vmap(lambda model, x: model(x), in_axes=(0, 2), out_axes=2)(
            self.temporal_transformer, x
        )

        res = h
        h = self.ln(h)
        h = self.mlp(h)

        return h + res


class LightSTFormerBlock(nnx.Module):

    def __init__(
        self,
        in_dim,
        out_dim,
        model_dim,
        num_heads,
        head_dim,
        num_chanels,
        *,
        rngs: nnx.Rngs
    ):

        self.spatial_transformer = Transformer(
            in_dim=model_dim,
            out_dim=model_dim,
            num_heads=num_heads,
            model_dim=model_dim,
            head_dim=head_dim,
            need_pos_enc=False,
            rngs=rngs,
        )

        self.ln = nnx.LayerNorm(model_dim, rngs=rngs)

        self.mlp = MLP(model_dim, model_dim, out_dim, rngs=rngs)

        backup = nnx.split_rngs(rngs, splits=num_chanels, only="params")

        self.temporal_transformer = Transformer(
            in_dim=in_dim,
            out_dim=model_dim,
            num_heads=num_heads,
            model_dim=model_dim,
            head_dim=head_dim,
            need_pos_enc=True,
            rngs=rngs,
        )

        nnx.restore_rngs(backup)

        self.rngs = nnx.Rngs(rngs.params())

    def __call__(self, x):
        B, S, C, D = x.shape

        x = x.transpose(0, 2, 1, 3).reshape(B * C, S, D)

        h = (
            self.temporal_transformer(x)
            .reshape(B, C, S, D)
            .transpose(0, 2, 1, 3)
            .reshape(B * S, C, D)
        )

        h = self.spatial_transformer(h).reshape(B, S, C, D)

        res = h
        h = self.ln(h)
        h = self.mlp(h)

        return h + res


class LightDiffGraphSTFormerBlock(nnx.Module):

    def __init__(
        self,
        in_dim,
        out_dim,
        model_dim,
        num_heads,
        head_dim,
        num_chanels,
        *,
        rngs: nnx.Rngs
    ):

        self.spatial_model = Grand(
            in_dim=model_dim,
            model_dim=model_dim,
            out_dim=model_dim,
            num_heads=num_heads,
            head_dim=head_dim,
            num_chanels=num_chanels,
            rngs=rngs,
        )

        self.ln = nnx.LayerNorm(model_dim, rngs=rngs)

        self.mlp = MLP(model_dim, model_dim, out_dim, rngs=rngs)

        backup = nnx.split_rngs(rngs, splits=num_chanels, only="params")

        self.temporal_transformer = Transformer(
            in_dim=in_dim,
            out_dim=model_dim,
            num_heads=num_heads,
            model_dim=model_dim,
            head_dim=head_dim,
            need_pos_enc=True,
            rngs=rngs,
        )

        nnx.restore_rngs(backup)

        self.rngs = nnx.Rngs(rngs.params())

    def __call__(self, x: jax.Array) -> jax.Array:
        B, S, C, D = x.shape

        x = x.transpose(0, 2, 1, 3).reshape(B * C, S, D)

        h = self.temporal_transformer(x).reshape(B, C, S, D).transpose(0, 2, 1, 3)

        t_grid = jnp.linspace(0, 0.5, 4)

        h = self.spatial_model(h, t_grid)

        res = h
        h = self.ln(h)
        out = self.mlp(h)

        return out + res


class LightTFormerBlock(nnx.Module):

    def __init__(
        self,
        in_dim,
        out_dim,
        model_dim,
        num_heads,
        head_dim,
        num_chanels,
        *,
        rngs: nnx.Rngs
    ):

        self.spatial_transformer = Transformer(
            in_dim=model_dim,
            out_dim=model_dim,
            num_heads=num_heads,
            model_dim=model_dim,
            head_dim=head_dim,
            need_pos_enc=False,
            rngs=rngs,
        )

        self.ln = nnx.LayerNorm(model_dim, rngs=rngs)

        self.mlp = MLP(model_dim, model_dim, out_dim, rngs=rngs)

        backup = nnx.split_rngs(rngs, splits=num_chanels, only="params")

        self.temporal_transformer = Transformer(
            in_dim=in_dim,
            out_dim=model_dim,
            num_heads=num_heads,
            model_dim=model_dim,
            head_dim=head_dim,
            need_pos_enc=True,
            rngs=rngs,
        )

        nnx.restore_rngs(backup)

        self.rngs = nnx.Rngs(rngs.params())

    def __call__(self, x):
        B, S, C, D = x.shape

        x = x.transpose(0, 2, 1, 3).reshape(B * C, S, D)

        h = self.temporal_transformer(x).reshape(B, C, S, D).transpose(0, 2, 1, 3)

        res = h
        h = self.ln(h)
        h = self.mlp(h)

        return h + res


class LightGConvSTFormerBlock(nnx.Module):

    def __init__(
        self,
        in_dim,
        out_dim,
        model_dim,
        num_heads,
        head_dim,
        num_chanels,
        *,
        rngs: nnx.Rngs
    ):

        self.spatial_model = TransformerConv(
            in_features=model_dim,
            out_features=model_dim,
            heads=num_heads,
            rngs=rngs,
        )

        self.ln = nnx.LayerNorm(model_dim, rngs=rngs)

        self.mlp = MLP(model_dim, model_dim, out_dim, rngs=rngs)

        self.temporal_transformer = Transformer(
            in_dim=in_dim,
            out_dim=model_dim,
            num_heads=num_heads,
            model_dim=model_dim,
            head_dim=head_dim,
            need_pos_enc=True,
            rngs=rngs,
        )

        self.rngs = nnx.Rngs(rngs.params())

    def __call__(
        self,
        x: jnp.ndarray,
        edge_index: jnp.ndarray,
    ) -> jax.Array:
        B, S, C, D = x.shape

        x = x.transpose(0, 2, 1, 3).reshape(B * C, S, D)

        h = (
            self.temporal_transformer(x)
            .reshape(B, C, S, D)
            .transpose(0, 2, 1, 3)
            .reshape(B * S, C, D)
        )

        h = self.spatial_model(h, edge_index)

        res = h
        h = self.ln(h)
        out = self.mlp(h)

        return out + res


In [6]:
from STFormer_ import VectorVAE
from flax import nnx


s = VectorVAE(4, 23, 3, rngs=nnx.Rngs(9))

print(s.vae.encoder)

Encoder( # Param: 6,969 (27.9 KB)
  logvar_proj=Linear( # Param: 1,656 (6.6 KB)
    bias=Param( # 69 (276 B)
      value=Array(shape=(3, 23), dtype=dtype('float32'))
    ),
    dot_general=<function dot_general at 0x113f7c460>,
    dtype=None,
    in_features=23,
    kernel=Param( # 1,587 (6.3 KB)
      value=Array(shape=(3, 23, 23), dtype=dtype('float32'))
    ),
    out_features=23,
    param_dtype=float32,
    precision=None,
    preferred_element_type=None,
    promote_dtype=<function promote_dtype at 0x1155f8ca0>,
    use_bias=True
  ),
  mu_proj=Linear( # Param: 1,656 (6.6 KB)
    bias=Param( # 69 (276 B)
      value=Array(shape=(3, 23), dtype=dtype('float32'))
    ),
    dot_general=<function dot_general at 0x113f7c460>,
    dtype=None,
    in_features=23,
    kernel=Param( # 1,587 (6.3 KB)
      value=Array(shape=(3, 23, 23), dtype=dtype('float32'))
    ),
    out_features=23,
    param_dtype=float32,
    precision=None,
    preferred_element_type=None,
    promote_dtype=<funct

In [13]:
from STFormer_ import VectorVAE

model = VectorVAE(
        in_dim=4,
        vae_latent=4,
        num_chanels=4,
        rngs=nnx.Rngs(0),
    )

In [16]:
mu, logvar = nnx.vmap(lambda model, h: model(h), in_axes=(0, 2), out_axes=(2, 2))(model.vae.encoder, jnp.ones((2, 4, 4, 4)))

In [17]:
mu.shape

(2, 4, 4, 4)

In [21]:
import flax.nnx as nnx
import jax
import jax.numpy as jnp
import optax

# ===== Модель =====
class SimpleModel(nnx.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, rngs):
        self.encoder = nnx.Linear(in_dim, hidden_dim, rngs=rngs)
        self.head = nnx.Linear(hidden_dim, out_dim, rngs=rngs)

    def __call__(self, x):
        x = self.encoder(x)
        x = jax.nn.relu(x)
        x = self.head(x)
        return x

# ===== Данные =====
key = jax.random.PRNGKey(0)
x = jax.random.normal(key, (128, 8))
y = jnp.sum(x, axis=-1, keepdims=True)

# ===== Инициализация =====
rngs = nnx.Rngs(0)
model = SimpleModel(8, 16, 1, rngs)

# ===== Loss =====
def loss_fn(model):
    pred = model(x)
    return jnp.mean((pred - y) ** 2)

@nnx.jit
def train_step(model, optimizer):
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)
    return loss

# ===== ЭТАП 1: обучаем всю модель =====
optimizer = nnx.Optimizer(
    model,
    optax.adam(1e-2),
    wrt=nnx.Param
)

for step in range(100):
    loss = train_step(model, optimizer)

print("After full training:", loss)

# ===== Делаем mask (замораживаем head) =====
params = nnx.state(model, nnx.Param)

def mask_fn(path, _):
    return path[0] != "head"   # head не обучается

mask = jax.tree_util.tree_map_with_path(mask_fn, params)

# ===== ЭТАП 2: новый optimizer с mask =====
optimizer = nnx.Optimizer(
    model,
    optax.masked(optax.adam(1e-2), mask),
    wrt=nnx.Param
)

# ===== Дообучаем только encoder =====
for step in range(100):
    loss = train_step(model, optimizer)

print("After freezing head:", loss)

After full training: 0.030601956


ValueError: Custom node type mismatch: expected type: <class 'flax.nnx.variablelib.Param'>, value: JitTracer(float32[16,1]).